# A11 · Fine-Tuning: SFT, LoRA & DPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a11_finetuning.ipynb)

Pre-trained models are adapted with supervised fine-tuning on instructions, made cheap with low-rank adapters, and aligned with preference optimisation. Derive LoRA's parameter savings, implement it from scratch, and compute the DPO loss by hand.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a11). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Explain SFT data formatting and loss masking on prompt tokens.
- Derive LoRA's trainable parameters $r(d_{in} + d_{out})$ and its update $\Delta W = \tfrac{\alpha}{r}BA$.
- Implement a LoRA layer from scratch and merge it back for zero-overhead inference.
- Compute the DPO loss for a preference pair.

## The math

### LoRA update and savings

$$h = W_0x + \frac{\alpha}{r}\,B A x, \qquad A\in\mathbb{R}^{r\times d_{in}},\; B\in\mathbb{R}^{d_{out}\times r}$$

Trainable parameters per adapted matrix: $r(d_{in} + d_{out})$ instead of $d_{in}d_{out}$.

**Worked example**

1. $d_{in} = d_{out} = 4096,\ r = 8$: $8\cdot(4096 + 4096) = 65{,}536$
2. full matrix: $4096^2 = 16{,}777{,}216$ (LoRA trains $0.39\%$ of it)
3. 7B model, adapters on Q and V in 32 layers: $2\cdot32\cdot65{,}536 = 4.2$M trainable (≈ 0.06% of the model)
4. optimizer state: $4.2\text{M}\cdot 16$ B ≈ 67 MB instead of 112 GB (the real reason LoRA fits on one GPU)

### Why B starts at zero

$$B_0 = 0 \;\Rightarrow\; h = W_0x \text{ at step 0}, \qquad \frac{\partial L}{\partial B} = \frac{\alpha}{r}\,\frac{\partial L}{\partial h}(Ax)^{\top} \neq 0$$

The model starts exactly as the base, yet $B$ still receives gradient because $A$ is random. $\alpha/r$ keeps the update scale stable when you change $r$.

**Worked example**

1. $\alpha = 16,\ r = 8 \Rightarrow$ scale 2 (common default)

### DPO loss

$$\mathcal{L}_{DPO} = -\log\sigma\!\Big(\beta\big[(\log\pi_\theta(y_w|x) - \log\pi_{ref}(y_w|x)) - (\log\pi_\theta(y_l|x) - \log\pi_{ref}(y_l|x))\big]\Big)$$

$y_w$ chosen, $y_l$ rejected; log-probs are summed over response tokens; $\beta$ controls how far the policy may drift from the reference.

**Worked example**

1. $\log\pi_\theta(y_w) = -12,\ \log\pi_{ref}(y_w) = -14$ (policy already likes the chosen answer more (+2))
2. $\log\pi_\theta(y_l) = -15,\ \log\pi_{ref}(y_l) = -13$ (and the rejected one less (−2))
3. margin $= \beta(2 - (-2)) = 0.1\cdot4 = 0.4$
4. $\mathcal{L} = -\log\sigma(0.4) = -\log 0.599 = 0.513$ (below $\log 2 = 0.693$, the loss at zero margin)

In [ ]:
%pip install -q transformers peft

## Lab: LoRA from scratch, then PEFT on a real model

Write LoRA yourself, verify zero-init and merging, then apply the Hugging Face PEFT library.

### Step 1

A LoRA wrapper around `nn.Linear`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                     # freeze W0 (and bias)
        self.A = nn.Parameter(torch.randn(r, base.in_features) / r)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale
    @torch.no_grad()
    def merged(self):
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        lin.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
        if self.base.bias is not None: lin.bias.copy_(self.base.bias)
        return lin

base = nn.Linear(4096, 4096)
lora = LoRALinear(base, r=8, alpha=16)
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(trainable, f"{trainable / base.weight.numel():.2%}")        # 65536 0.39%
x = torch.randn(3, 4096)
print(torch.allclose(lora(x), base(x)))                         # True: B = 0 at init

### Step 2

Train the adapter to imitate a target layer, then merge and check equivalence.

In [ ]:
torch.manual_seed(0)
target = nn.Linear(256, 256)
lora = LoRALinear(nn.Linear(256, 256), r=4, alpha=8)
opt = torch.optim.AdamW([lora.A, lora.B], lr=1e-2)
for step in range(500):
    x = torch.randn(64, 256)
    loss = F.mse_loss(lora(x), target(x).detach())
    opt.zero_grad(); loss.backward(); opt.step()
print("loss", loss.item())                     # a rank-4 update can only partly match a full-rank change
merged = lora.merged()
x = torch.randn(5, 256)
print(torch.allclose(lora(x), merged(x), atol=1e-5))   # True: zero inference overhead after merging

### Step 3

The DPO loss, checked against the worked example.

In [ ]:
def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1):
    # each argument: summed log-prob of the response tokens
    logits = beta * ((pi_w - ref_w) - (pi_l - ref_l))
    return -F.logsigmoid(logits).mean()
print(dpo_loss(torch.tensor(-12.), torch.tensor(-15.), torch.tensor(-14.), torch.tensor(-13.)))   # 0.5130

### Step 4

PEFT on GPT-2 (`pip install transformers peft`). The same few lines apply to Llama, Qwen or Mistral.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
model = AutoModelForCausalLM.from_pretrained("gpt2")
cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["c_attn"],
                 fan_in_fan_out=True, task_type="CAUSAL_LM")   # GPT-2 stores c_attn transposed
model = get_peft_model(model, cfg)
model.print_trainable_parameters()      # ~0.24% of 124M

### Clean up

Disconnect the runtime. Adapter weights are only a few MB, so save them if you want to reuse them.

## Check yourself

**1. LoRA rank 16 on a $4096\times11008$ MLP matrix. Trainable parameters?**

A. 241,664  
B. 45M  
C. 65,536  
D. 1.2M

<details><summary>Answer</summary>

**A.** $16\cdot(4096 + 11008) = 16\cdot15{,}104 = 241{,}664$.

</details>

**2. Why is $B$ initialised to zero?**

A. To save memory  
B. So the adapted model starts identical to the base model  
C. To prevent overfitting  
D. Because A is zero too

<details><summary>Answer</summary>

**B.** $BA = 0$ at step 0, while $B$ still gets non-zero gradient because $A$ is random.

</details>

**3. In DPO, a margin of 0 gives a loss of…**

A. 0  
B. $\log 2 \approx 0.693$  
C. 1  
D. $\infty$

<details><summary>Answer</summary>

**B.** $-\log\sigma(0) = -\log 0.5 = \log 2$.

</details>